# 03｜Random Forest 異常分類訓練

本 notebook 使用 `data/processed/fccl_cleaned.csv` 訓練 Random Forest 分類器，以 `Injected_Anomaly` 作為目標，預測批次是否為注入異常（0＝正常、1＝異常）。

流程包含分層訓練/測試切分、數值與類別欄位前處理、交叉驗證隨機搜尋、獨立測試集評估，以及模型匯出。資料集只有 492 筆且類別不平衡，因此使用 F1 作為主要調參指標，並同時檢視 precision、recall、ROC-AUC 與 PR-AUC。

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, make_scorer)
from sklearn.model_selection import (train_test_split, RandomizedSearchCV,
                                     StratifiedKFold, cross_validate)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from matplotlib import font_manager

# 尋找輸入檔；輸出模型存放在專案 models 目錄。
input_name = "fccl_cleaned.csv"
input_path = next((p / "data" / "processed" / input_name for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "processed" / input_name).exists()), None)
if input_path is None:
    raise FileNotFoundError("找不到 data/processed/fccl_cleaned.csv，請先執行 02_data_preparation.ipynb。")
project_dir = input_path.parents[2]
model_dir = project_dir / "models"
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / "random_forest_anomaly.joblib"

# 圖表中文使用系統中文字型；若找不到則顯示安裝提示。
font_candidates = ["Microsoft JhengHei", "Microsoft YaHei", "Noto Sans CJK TC", "Noto Sans CJK SC", "PingFang TC", "SimHei"]
installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next((name for name in font_candidates if name in installed_fonts), None)
if chinese_font:
    plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
else:
    warnings.warn("找不到中文字型，圖表中文可能顯示為方框。請安裝 Microsoft JhengHei 或 Noto Sans CJK TC。")
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid")
print(f"資料來源：{input_path}")
print(f"模型輸出：{model_path}")

## 1. 載入資料與定義預測目標

先確認目標標籤及類別比例。資料中 `QC_Result`、`Anomaly_Pattern` 和 `Potential_Root_Cause_Synthetic` 是異常結果或描述，會排除以避免模型直接讀到答案；批次 ID、日期、月份和序號也排除，以降低記憶批次或時間順序的風險。厚度量測與製程參數則作為可用特徵。

In [ ]:
df = pd.read_csv(input_path)
target_col = "Injected_Anomaly"
if target_col not in df.columns:
    raise ValueError(f"找不到目標欄位：{target_col}")

# 排除標籤本身、事後 QC/異常描述，以及僅作識別或時間索引的欄位。
excluded_cols = [
    "Injected_Anomaly", "QC_Result", "Anomaly_Pattern", "Potential_Root_Cause_Synthetic",
    "Batch_ID", "Sample_Date", "Month", "Model_Batch_Sequence",
]
X = df.drop(columns=[c for c in excluded_cols if c in df.columns]).copy()
y = df[target_col].astype(int)
if y.nunique() != 2:
    raise ValueError("目標欄位必須包含兩個類別，且以 0/1 編碼。")

# 自動依資料型態分出數值和類別特徵，並記錄欄位供模型推論時重用。
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()
print(f"樣本數：{len(df)}；特徵數：{X.shape[1]}")
print("類別比例：")
display(y.value_counts().sort_index().rename(index={0: "正常（0）", 1: "異常（1）"}).to_frame("筆數"))
print("數值特徵：", numeric_features)
print("類別特徵：", categorical_features)

## 2. 分層切分與前處理

保留 20% 作為最後一次評估的測試集，並以分層抽樣維持異常比例。所有缺值填補與類別編碼都放在 Pipeline 中，只會在每個訓練折內學習，避免驗證資料資訊滲入前處理。

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
], remainder="drop")

pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("classifier", RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=1)),
])
print(f"訓練集：{len(X_train)} 筆；測試集：{len(X_test)} 筆")
print(f"訓練集異常比例：{y_train.mean():.1%}；測試集異常比例：{y_test.mean():.1%}")

## 3. Random Forest 參數搜尋

使用 5 折分層交叉驗證，以異常類別的 F1 分數作為選模依據。參數搜尋只使用訓練集；測試集不會參與調參。

In [ ]:
param_distributions = {
    "classifier__n_estimators": [200, 400, 600],
    "classifier__max_depth": [None, 5, 10, 20],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": ["sqrt", 0.5, 1.0],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=24,
    scoring=make_scorer(f1_score, pos_label=1),
    cv=cv,
    random_state=42,
    # Windows/Jupyter 環境使用單一程序，避免 loky 子程序反序列化失敗。
    n_jobs=1,
    refit=True,
    return_train_score=True,
    verbose=1,
)
search.fit(X_train, y_train)
best_model = search.best_estimator_
print("最佳交叉驗證 F1：", round(search.best_score_, 4))
print("最佳參數：")
display(pd.Series(search.best_params_, name="最佳設定").to_frame())
search_results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
display(search_results[["rank_test_score", "mean_test_score", "std_test_score", "mean_train_score", "params"]].head(10))

## 4. 獨立測試集評估

使用最佳模型在保留測試集上評估。異常類別為正類（1）；除分類報告與混淆矩陣外，也查看 ROC-AUC 和 PR-AUC。PR-AUC 對異常樣本較少的資料特別有參考價值。

In [ ]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]
metrics = pd.Series({
    "Precision（異常）": precision_score(y_test, y_pred, zero_division=0),
    "Recall（異常）": recall_score(y_test, y_pred, zero_division=0),
    "F1（異常）": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob),
    "PR-AUC（Average Precision）": average_precision_score(y_test, y_prob),
}, name="測試集分數")
display(metrics.to_frame().round(4))
print(classification_report(y_test, y_pred, labels=[0, 1], target_names=["正常（0）", "異常（1）"], zero_division=0))

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=[0, 1], display_labels=["正常", "異常"], cmap="Blues", colorbar=False, ax=ax)
ax.set(title="測試集混淆矩陣", xlabel="模型預測", ylabel="實際標籤")
plt.tight_layout()
plt.show()

## 5. 匯出模型與評估摘要

將已完成前處理與訓練的 Pipeline、推論所需特徵欄位、目標定義、排除欄位及測試集指標一併存入 Joblib 檔。載入後以相同欄位順序傳入特徵資料即可預測。模型只適用於欄位語意及量測流程相同的資料；由於訓練資料是合成資料，實際部署前應以真實且具代表性的資料重新驗證。

In [ ]:
model_bundle = {
    "model": best_model,
    "feature_columns": X.columns.tolist(),
    "target_column": target_col,
    "target_mapping": {0: "正常", 1: "異常"},
    "excluded_columns": [c for c in excluded_cols if c in df.columns],
    "best_params": search.best_params_,
    "cv_best_f1": float(search.best_score_),
    "test_metrics": {name: float(value) for name, value in metrics.items()},
    "random_state": 42,
}
joblib.dump(model_bundle, model_path, compress=3)
print(f"模型已匯出：{model_path}")
print("使用方式範例：")
print("bundle = joblib.load(model_path)")
print("predictions = bundle['model'].predict(new_data[bundle['feature_columns']])")

# 顯示模型使用的特徵欄位，方便追蹤訓練輸入。
display(pd.DataFrame({"模型特徵欄位": X.columns}))

## 結果解讀提醒

- 調參的最佳分數是訓練集 5 折交叉驗證 F1；測試集指標是保留資料上的獨立估計。
- 異常類別比例較低，應同時關注 precision 和 recall，依漏報與誤報成本決定後續分類門檻。
- `QC_Result`、異常模式與合成根因已排除，避免答案洩漏；批次 ID、日期和月份也不作為輸入。
- 輸出模型：`models/random_forest_anomaly.joblib`。